# Ingestão Bronze - Base dos Dados → S3

Objetivo: extrair as 6 fontes confirmadas na etapa de descoberta (`01_descoberta_fontes.ipynb`)
do BigQuery e gravar como Parquet particionado no S3, na camada Bronze do data lake.

**Origem:** `basedosdados.br_inep_avaliacao_alfabetizacao` (BigQuery)

**Destino:** `s3://brazil-literacy-lakehouse-joaopaulo/bronze/`

In [1]:
import basedosdados as bd
import pandas as pd

bd.config.billing_project_id = "brazil-literacy-lakehouse"

DATASET_ID = "br_inep_avaliacao_alfabetizacao"

FONTES = {
    "uf": "uf",
    "municipio": "municipio",
    "meta_alfabetizacao_brasil": "meta_alfabetizacao_brasil",
    "meta_alfabetizacao_uf": "meta_alfabetizacao_uf",
    "meta_alfabetizacao_municipio": "meta_alfabetizacao_municipio",
    "alunos": "alunos",
}

In [2]:
import awswrangler as wr

BUCKET = "brazil-literacy-lakehouse-joaopaulo"

query = f"""
SELECT *
FROM `basedosdados.{DATASET_ID}.meta_alfabetizacao_brasil`
"""
df = bd.read_sql(query, billing_project_id="brazil-literacy-lakehouse")
df

Downloading: 100%|██████████|


,ano,rede,taxa_alfabetizacao,meta_alfabetizacao_2024,meta_alfabetizacao_2025,meta_alfabetizacao_2026,meta_alfabetizacao_2027,meta_alfabetizacao_2028,meta_alfabetizacao_2029,meta_alfabetizacao_2030,percentual_participacao
0,2025,Pública,66.0,60.0,64.00,67.00,71.00,74.00,77.00,80.0,88.00
1,2024,Pública,59.2,59.9,63.77,67.47,70.97,74.23,77.24,80.0,87.37
2,2023,Pública,55.9,59.9,63.77,67.47,70.97,74.23,77.24,80.0,86.00


In [3]:
import awswrangler as wr

BUCKET = "brazil-literacy-lakehouse-joaopaulo"

wr.s3.to_parquet(
    df=df,
    path=f"s3://{BUCKET}/bronze/meta_alfabetizacao_brasil/",
    dataset=True,
    partition_cols=["ano"],
)

{'paths': ['s3://brazil-literacy-lakehouse-joaopaulo/bronze/meta_alfabetizacao_brasil/ano=2023/82f6504c4fc34f85807307b9b31ef9d4.snappy.parquet',
  's3://brazil-literacy-lakehouse-joaopaulo/bronze/meta_alfabetizacao_brasil/ano=2024/82f6504c4fc34f85807307b9b31ef9d4.snappy.parquet',
  's3://brazil-literacy-lakehouse-joaopaulo/bronze/meta_alfabetizacao_brasil/ano=2025/82f6504c4fc34f85807307b9b31ef9d4.snappy.parquet'],
 'partitions_values': {'s3://brazil-literacy-lakehouse-joaopaulo/bronze/meta_alfabetizacao_brasil/ano=2023/': ['2023'],
  's3://brazil-literacy-lakehouse-joaopaulo/bronze/meta_alfabetizacao_brasil/ano=2024/': ['2024'],
  's3://brazil-literacy-lakehouse-joaopaulo/bronze/meta_alfabetizacao_brasil/ano=2025/': ['2025']}}

In [4]:
query = f"""
SELECT *
FROM `basedosdados.{DATASET_ID}.alunos`
LIMIT 5
"""
bd.read_sql(query, billing_project_id="brazil-literacy-lakehouse")

Downloading: 100%|██████████|


,ano,id_municipio,id_escola,id_aluno,caderno,serie,rede,presenca,preenchimento_caderno,alfabetizado,proficiencia,peso_aluno
0,2023,1101492,60000268,11017171,1,2,3,0,0,0,NaN,NaN
1,2023,1300300,60000545,13001616,1,2,3,0,0,0,NaN,NaN
2,2023,1302603,60000636,13028775,1,2,3,0,0,0,NaN,NaN
3,2023,1302603,60001177,13033437,1,2,3,0,0,0,NaN,NaN
4,2023,1506807,60001474,15104887,1,2,3,0,0,0,NaN,NaN


In [5]:
resultados_bronze = {}

for nome, tabela in FONTES.items():
    print(f"Extraindo {nome}...")

    query = f"""
    SELECT *
    FROM `basedosdados.{DATASET_ID}.{tabela}`
    """
    df = bd.read_sql(query, billing_project_id="brazil-literacy-lakehouse")

    resposta = wr.s3.to_parquet(
        df=df,
        path=f"s3://{BUCKET}/bronze/{nome}/",
        dataset=True,
        partition_cols=["ano"],
    )

    resultados_bronze[nome] = resposta
    print(f"  -> {len(df):,} linhas, {len(resposta['paths'])} arquivo(s) gravado(s)")

Extraindo uf...
Downloading: 100%|██████████|
  -> 145 linhas, 2 arquivo(s) gravado(s)
Extraindo municipio...
Downloading: 100%|██████████|
  -> 23,995 linhas, 2 arquivo(s) gravado(s)
Extraindo meta_alfabetizacao_brasil...
Downloading: 100%|██████████|
  -> 3 linhas, 3 arquivo(s) gravado(s)
Extraindo meta_alfabetizacao_uf...
Downloading: 100%|██████████|
  -> 81 linhas, 3 arquivo(s) gravado(s)
Extraindo meta_alfabetizacao_municipio...
Downloading: 100%|██████████|
  -> 10,704 linhas, 2 arquivo(s) gravado(s)
Extraindo alunos...
Downloading: 100%|██████████|
  -> 3,867,999 linhas, 2 arquivo(s) gravado(s)
